# Inspect One Masked Event Batch - v3

This notebook mirrors the v14 inspection flow for the masked event model: load one event-chunk batch, check tensor shapes and finite values, restore a checkpoint if provided, render Keras-style model summary/graph artifacts, run one masked-autoencoder inference, and decode one forecast table.

In [ ]:
from pathlib import Path

MODEL_VERSION = "v3"
MODEL_ROOT = Path(r"D:\TradingData\quant-research-workbench\market_data\models\masked_event_model") / MODEL_VERSION

# Edit these values for the batch/checkpoint you want to inspect.
WORKSPACE = None  # Leave None to auto-detect local repo or workstation runtime root.
CACHE_ROOT = Path(r"D:\market-data\flatfiles\us_stocks_sip\derived\event_chunks_v2")
CANONICAL_ROOT = Path(r"D:\market-data\flatfiles\us_stocks_sip\derived\canonical_events_v2")
SESSION_DATE = "2025-11-03"
TICKERS = ("AAPL",)  # Use ("ALL",) only after checking memory and runtime.
BATCH_SIZE = 8
CONTEXT_SECONDS = 30
CHUNK_MS = 500
DEVICE = "cuda"  # Use "cpu" if CUDA is unavailable.
USE_AMP = False  # Keep False while checking exact tensors/losses.
LOAD_CHECKPOINT_CONFIG = True  # Use checkpoint model/data shape when CHECKPOINT_PATH is set.

# Paste a checkpoint path, or leave empty to auto-load the latest v3 checkpoint_last.pt under MODEL_ROOT.
CHECKPOINT_PATH = ""
# Example:
# CHECKPOINT_PATH = str(MODEL_ROOT / "mem-v3-d384-emb256-e2-t8-d4-mask70-chunk500-nov2025" / "checkpoint_last.pt")

ARCHITECTURE_OUTPUT_DIR = MODEL_ROOT / "inspect_architecture"

In [ ]:
import sys
import math
import json
from dataclasses import fields
from pathlib import Path

import numpy as np
import polars as pl
import torch
from torch.utils.data import DataLoader


def resolve_workspace(explicit):
    if explicit:
        path = Path(explicit)
        if (path / "research" / "masked_event_model" / MODEL_VERSION).exists():
            return path
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates.extend([
        Path(r"D:\TradingCodes\quant-research-workbench"),
        Path(r"D:\TradingCodes\quant-research-workbench-masked-event-v3-runtime"),
    ])
    for path in candidates:
        if (path / "research" / "masked_event_model" / MODEL_VERSION).exists():
            return path
    raise FileNotFoundError("Could not find a workspace root containing research/masked_event_model/v3.")


WORKSPACE = resolve_workspace(WORKSPACE)
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from research.masked_event_model.v3.config import DataConfig, ExperimentConfig, LossConfig, MaskConfig, ModelConfig
from research.masked_event_model.v3.data import EventChunkDataset, discover_chunk_files, parse_tickers, target_horizons_from_columns
from research.masked_event_model.v3.losses import masked_autoencoder_loss
from research.masked_event_model.v3.masking import build_structured_masks
from research.masked_event_model.v3.model import MaskedEventAutoencoder
from research.masked_event_model.v3.model_artifacts import save_model_architecture_artifacts
from research.masked_event_model.v3.schema import CHUNK_SUMMARY_COLUMNS, QUOTE_FEATURE_COLUMNS, TRADE_FEATURE_COLUMNS
from research.masked_event_model.v3.targets import decode_binary_magnitude_logits_to_bps

np.set_printoptions(precision=5, suppress=True)
torch.set_printoptions(precision=5, sci_mode=False)
torch.set_float32_matmul_precision("high")


def dataclass_from_dict(cls, payload, tuple_fields=()):
    allowed = {field.name for field in fields(cls)}
    values = {key: value for key, value in payload.items() if key in allowed}
    for name in ("cache_root", "canonical_root", "output_root"):
        if name in values:
            values[name] = Path(values[name])
    for name in tuple_fields:
        if name in values:
            values[name] = tuple(values[name])
    return cls(**values)


def latest_checkpoint(model_root):
    candidates = list(model_root.glob("*/checkpoint_last.pt")) + list(model_root.glob("*/last.pt")) + list(model_root.glob("*/best.pt"))
    return max(candidates, key=lambda path: path.stat().st_mtime) if candidates else None


checkpoint = None
checkpoint_path = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else latest_checkpoint(MODEL_ROOT)
if checkpoint_path:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    checkpoint_config = checkpoint.get("config", {}) if isinstance(checkpoint, dict) else {}
else:
    checkpoint_config = {}
checkpoint_version = checkpoint_config.get("experiment_version") or checkpoint_config.get("version")
if checkpoint_version and checkpoint_version != MODEL_VERSION:
    raise ValueError(f"Checkpoint version {checkpoint_version!r} does not match notebook version {MODEL_VERSION!r}.")

if checkpoint_config and LOAD_CHECKPOINT_CONFIG:
    data_config = dataclass_from_dict(DataConfig, checkpoint_config.get("data", {}), tuple_fields=("tickers",))
    model_config = dataclass_from_dict(ModelConfig, checkpoint_config.get("model", {}))
    mask_config = dataclass_from_dict(MaskConfig, checkpoint_config.get("masks", {}))
    loss_config = dataclass_from_dict(LossConfig, checkpoint_config.get("losses", {}))
    print("using data/model shape from checkpoint config")
else:
    data_config = DataConfig(
        cache_root=CACHE_ROOT,
        canonical_root=CANONICAL_ROOT,
        train_start_date=SESSION_DATE,
        train_end_date=SESSION_DATE,
        validation_start_date=SESSION_DATE,
        validation_end_date=SESSION_DATE,
        test_start_date=SESSION_DATE,
        test_end_date=SESSION_DATE,
        tickers=parse_tickers(TICKERS),
        context_seconds=CONTEXT_SECONDS,
        chunk_ms=CHUNK_MS,
        shuffle_files=False,
        shuffle_windows=False,
    )
    model_config = ModelConfig()
    mask_config = MaskConfig()
    loss_config = LossConfig()

# These are inspection controls, so they intentionally override checkpoint split/ticker settings.
data_config.cache_root = CACHE_ROOT
data_config.canonical_root = CANONICAL_ROOT
data_config.train_start_date = SESSION_DATE
data_config.train_end_date = SESSION_DATE
data_config.validation_start_date = SESSION_DATE
data_config.validation_end_date = SESSION_DATE
data_config.test_start_date = SESSION_DATE
data_config.test_end_date = SESSION_DATE
data_config.tickers = parse_tickers(TICKERS)
data_config.max_files = 0
data_config.max_windows_per_file = max(BATCH_SIZE, data_config.max_windows_per_file)
data_config.shuffle_files = False
data_config.shuffle_windows = False
experiment_config = ExperimentConfig(data=data_config, masks=mask_config, model=model_config, losses=loss_config)

device = torch.device(DEVICE if DEVICE == "cpu" or torch.cuda.is_available() else "cpu")
print(f"workspace={WORKSPACE}")
print(f"device={device}")
print(f"cache_root={data_config.cache_root}")
print(f"tickers={data_config.tickers} session={SESSION_DATE}")
print(f"context_seconds={data_config.context_seconds} chunk_ms={data_config.chunk_ms} context_chunks={data_config.context_chunks}")
print(f"max_quote_events={data_config.max_quote_events} max_trade_events={data_config.max_trade_events} max_total_events={data_config.max_total_events}")
print(f"features quote={len(QUOTE_FEATURE_COLUMNS)} trade={len(TRADE_FEATURE_COLUMNS)} summary={len(CHUNK_SUMMARY_COLUMNS)} target_bits={data_config.target_bit_count}")

In [ ]:
files = discover_chunk_files(data_config, start_date=SESSION_DATE, end_date=SESSION_DATE, tickers=data_config.tickers)
if not files:
    raise FileNotFoundError(f"No chunk files found under {data_config.cache_root} for {SESSION_DATE} and tickers={data_config.tickers}")

schema = pl.scan_parquet(str(files[0].path)).collect_schema()
target_horizons = target_horizons_from_columns(schema.names())
horizon_count = len(target_horizons)
print("chunk files:", len(files))
print("first file:", files[0].path)
print("target horizons:", target_horizons)

# Keep one deterministic batch for inspection.
dataset = EventChunkDataset(
    config=data_config,
    split="train",
    batch_size=BATCH_SIZE,
    seed=17,
)
loader = DataLoader(dataset, batch_size=None, num_workers=0)
batch = next(iter(loader))
print("batch keys:", sorted(batch.keys()))
for key, value in batch.items():
    if torch.is_tensor(value):
        finite = bool(torch.isfinite(value).all()) if value.is_floating_point() else True
        if value.numel() and value.is_floating_point():
            min_value = float(value.min())
            max_value = float(value.max())
        else:
            min_value = math.nan
            max_value = math.nan
        print(f"{key:24s} shape={tuple(value.shape)} dtype={value.dtype} finite={finite} min={min_value:.6g} max={max_value:.6g}")
    else:
        print(f"{key:24s} type={type(value).__name__} len={len(value) if hasattr(value, '__len__') else 'n/a'} sample={value[:5] if isinstance(value, list) else value}")

In [ ]:
batch["targets"]

In [ ]:
# Inspect the first sample's latest context chunk after input normalization.
row = 0
latest_chunk = -1
print("first sample ticker:", batch.get("ticker", [None])[row])
print("origin_timestamp_ns:", int(batch["origin_timestamp_ns"][row]))
print("current_mid:", float(batch["current_mid"][row]))
print("target_bps:", batch["target_bps"][row].numpy())

summary_frame = pl.DataFrame({"summary_feature": list(CHUNK_SUMMARY_COLUMNS), "normalized_value_at_context_end": batch["chunk_summary"][row, latest_chunk].numpy()})
quote_values = batch["quote_values"][row, latest_chunk].numpy()
trade_values = batch["trade_values"][row, latest_chunk].numpy()
quote_nonzero = np.where(np.abs(quote_values).sum(axis=1) > 0.0)[0]
trade_nonzero = np.where(np.abs(trade_values).sum(axis=1) > 0.0)[0]
print("latest chunk non-empty quote events:", len(quote_nonzero), "trade events:", len(trade_nonzero))
display(summary_frame)
if len(quote_nonzero):
    display(pl.DataFrame(quote_values[quote_nonzero[:10]], schema=list(QUOTE_FEATURE_COLUMNS), orient="row"))
if len(trade_nonzero):
    display(pl.DataFrame(trade_values[trade_nonzero[:10]], schema=list(TRADE_FEATURE_COLUMNS), orient="row"))

In [ ]:
model = MaskedEventAutoencoder(
    quote_feature_count=len(QUOTE_FEATURE_COLUMNS),
    trade_feature_count=len(TRADE_FEATURE_COLUMNS),
    summary_feature_count=len(CHUNK_SUMMARY_COLUMNS),
    context_chunks=data_config.context_chunks,
    max_quote_events=data_config.max_quote_events,
    max_trade_events=data_config.max_trade_events,
    max_total_events=data_config.max_total_events,
    horizon_count=horizon_count,
    target_bit_count=data_config.target_bit_count,
    config=model_config,
).to(device)

if checkpoint is not None:
    state_dict = checkpoint.get("model", checkpoint.get("model_state_dict", checkpoint))
    model.load_state_dict(state_dict, strict=True)
    print(f"loaded checkpoint: {checkpoint_path}")
    print("checkpoint step:", checkpoint.get("step"), "best_score:", checkpoint.get("best_score"))
else:
    print("CHECKPOINT_PATH is empty and no latest checkpoint was found; using fresh randomly initialized model.")

param_count = sum(parameter.numel() for parameter in model.parameters())
trainable_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(f"model parameters={param_count:,} trainable={trainable_count:,}")

In [ ]:
# Keras-style model summary and graph view.
# Optional packages for richer output:
#   pip install torchinfo torchview graphviz
# Graph rendering also needs the Graphviz dot executable on PATH.

from IPython.display import Markdown, Image, display

SUMMARY_BATCH_SIZE = 1
SUMMARY_DEPTH = 8
GRAPH_DEPTH = 3

ARCHITECTURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
architecture_info = save_model_architecture_artifacts(
    model=model,
    data_config=data_config,
    output_dir=ARCHITECTURE_OUTPUT_DIR,
    version=f"mem-{MODEL_VERSION}-inspect",
    torch_module=torch,
    wandb_run=None,
    summary_batch_size=SUMMARY_BATCH_SIZE,
    summary_depth=SUMMARY_DEPTH,
    graph_depth=GRAPH_DEPTH,
)

display(Markdown(f"### {MODEL_VERSION} Model Summary"))
print("architecture_dir:", architecture_info.get("architecture_dir"))
print("Forward input shapes:", architecture_info.get("input_shapes"))
print("summary:", architecture_info.get("summary_path"))
print("graph png:", architecture_info.get("graph_png_path"))
print("graph svg:", architecture_info.get("graph_svg_path"))
print("graph pdf:", architecture_info.get("graph_pdf_path"))
print("errors:", architecture_info.get("errors"))

summary_path = Path(architecture_info["summary_path"])
if summary_path.exists():
    text = summary_path.read_text(encoding="utf-8")
    display(Markdown("```text\n" + text[:30000] + ("\n... truncated ..." if len(text) > 30000 else "") + "\n```"))

png_path = architecture_info.get("graph_png_path")
if png_path and Path(png_path).exists():
    display(Image(filename=str(png_path)))

In [ ]:
def move_tensor_batch(batch_dict, device):
    return {key: value.to(device, non_blocking=True) if torch.is_tensor(value) else value for key, value in batch_dict.items()}

model.eval()
device_batch = move_tensor_batch(batch, device)
masks = build_structured_masks(
    quote_values=device_batch["quote_values"],
    trade_values=device_batch["trade_values"],
    chunk_summary=device_batch["chunk_summary"],
    event_kinds=device_batch["event_kinds"],
    config=mask_config,
)
with torch.inference_mode():
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP and device.type == "cuda"):
        output = model(
            device_batch["quote_values"],
            device_batch["trade_values"],
            device_batch["event_kinds"],
            device_batch["event_indices"],
            device_batch["chunk_summary"],
            masks,
        )
        loss, loss_parts = masked_autoencoder_loss(output, device_batch, masks, loss_config)

print("forecast_logits shape:", tuple(output.forecast_logits.shape))
print("embedding shape:", tuple(output.embeddings.shape))
print("encoded_chunks shape:", tuple(output.encoded_chunks.shape))
print("loss:", float(loss.detach().cpu()))
print("loss_parts:", json.dumps(loss_parts, indent=2, sort_keys=True))
print("prediction finite:", bool(torch.isfinite(output.forecast_logits).all()))
print("target finite:", bool(torch.isfinite(device_batch["targets"]).all()))

In [ ]:
forecast_logits = output.forecast_logits.detach().cpu().numpy()
pred_bps = decode_binary_magnitude_logits_to_bps(forecast_logits).reshape(batch["target_bps"].shape)
target_bps = batch["target_bps"].numpy()
current_mid = batch["current_mid"].numpy().reshape(-1, 1)
pred_mid = current_mid * np.exp(pred_bps[:, :, 0] / 10000.0)
target_mid = current_mid * np.exp(target_bps[:, :, 0] / 10000.0)

rows = []
row_count = min(len(batch.get("ticker", [])), pred_bps.shape[0], 20)
for idx in range(row_count):
    for h_idx, horizon in enumerate(target_horizons):
        rows.append({
            "row": idx,
            "ticker": batch.get("ticker", [None] * BATCH_SIZE)[idx],
            "origin_timestamp_ns": int(batch["origin_timestamp_ns"][idx]),
            "horizon_chunks": int(horizon),
            "current_mid": float(batch["current_mid"][idx]),
            "target_mid": float(target_mid[idx, h_idx]),
            "prediction_mid": float(pred_mid[idx, h_idx]),
            "target_bps": float(target_bps[idx, h_idx, 0]),
            "prediction_bps": float(pred_bps[idx, h_idx, 0]),
            "abs_error_bps": abs(float(pred_bps[idx, h_idx, 0] - target_bps[idx, h_idx, 0])),
            "dir_correct": bool((pred_bps[idx, h_idx, 0] > 0.0) == (target_bps[idx, h_idx, 0] > 0.0)),
        })

pl.DataFrame(rows)